# Class 5. SQL 

## Plan:

<h2>

1) What is SQL? <br>

2) How to start working in SQL via python? <br>

3) Connecting to PostgreSQL <br>

4) Types of data in SQL <br>

5) SQL hierarchy and object creation/ deletion <br>

6) Filling SQL tables <br>

7) Select data from SQL <br>

8) Common attributes <br>

9) Повторение- мать ученья <br>

</h2>

## Imports

In [1]:
#in case you didn't have it installed
!pip install -q psycopg2 --user

In [14]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT, ISOLATION_LEVEL_SERIALIZABLE # to create databases
from psycopg2.extras import execute_batch

import pandas as pd

## What is SQL?

SQL (Structured Query Language) is a domain-specific language for managing and querying relational databases, enabling structured data storage, retrieval, and manipulation. It's essential for handling large datasets efficiently in applications like finance, web services, and analytics, where data integrity and relationships matter.

## How to start working in SQL via python?

In our course we'll be working with the PostgreSQL distribution of SQL. When you're going to be downloading it ***please remember your user, password and port***, the download link is [here](https://www.postgresql.org/download/). After you download it, you can close it (no need to install additional stuff)

## Connecting to PostgreSQL

It is ***very*** unsafe to leave your password hardcoded or anywhere in the code so store it in a separate file to avoid having your account breached

In [ ]:
with open('pwd.txt', 'w') as file:
    file.write(input('Enter the password: ')) #put here the password you set during the installation

In [15]:
with open('pwd.txt', 'r') as file:
    pwd = file.read()

usr_nm = 'postgres' #this is the default name, replace it with yours if you've changed the name
port = '5432' #this is the default port, replace with your port if you've changed it

# Connection details
DB_NAME = "postgres"
DB_USER = usr_nm
DB_PASS = pwd # The password you set during installation
DB_HOST = "localhost" # For now because you're going to be running the scripts and SQL locally
DB_PORT = port

try:
    # Connect to the PostgreSQL database server
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        host=DB_HOST,
        port=DB_PORT
    )

except psycopg2.Error as e:
    print(f"Error connecting to the database: {e}")

else:
    print('Connection successful!')

finally:
    conn.close()

Connection successful!


## Types of data in SQL

|Category | Data Types | Description
|---------|------------|------------|
|Numeric | INTEGER, BIGINT, DECIMAL, NUMERIC, REAL | Whole numbers, fixed‑point, floating‑point |
|Character | VARCHAR(n), CHAR(n), TEXT | Variable‑length strings, fixed‑length, unlimited
|Date/Time	| DATE, TIME, TIMESTAMP, INTERVAL | Dates, times, timestamps, time intervals
|Boolean | BOOLEAN | True/false
|Binary | BYTEA | Binary data (e.g., files)
|JSON | JSON, JSONB | JSON data (JSONB is binary, more efficient)
|Array | INTEGER[], TEXT[], etc. | Array of values
|UUID | UUID | Universally Unique Identifiers
|Geometric | POINT, LINE, etc. | Spatial data

When creating a table, you assign a type to each column to enforce data integrity and optimize storage.\
During our course we will be looking only at `Numeric`, `Character`, `Date/Time` and `Boolean` data types

## SQL hierarchy and object creation

![image](hierarchy.png)

### Creating/Deleting a database

In [26]:
NEW_DB_NAME = 'test' #the name of the database to be created

# Connect to default 'postgres' database first
conn = psycopg2.connect(
    host="localhost",
    database="postgres",  # Connect to default database
    user=usr_nm,
    password=pwd
)

conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

# conn.commit()
# conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)
# Create the 'test' database
try:
    cur.execute(f"CREATE DATABASE {NEW_DB_NAME};")
    print('SUCCESS')
    conn.set_isolation_level(ISOLATION_LEVEL_SERIALIZABLE)
    cur.execute(f"CREATE DATABASE {NEW_DB_NAME}_1;")
except Exception as e:
    print('Database either exists or there are some serious troubles')
    print(str(e))
else:
    print('Database created successfully!')
finally:
    cur.close()
    conn.close()

SUCCESS
Database either exists or there are some serious troubles
ОШИБКА:  CREATE DATABASE не может выполняться внутри блока транзакции



In [25]:
NEW_DB_NAME = 'test' #the name of the database to be created

# Connect to default 'postgres' database first
conn = psycopg2.connect(
    host="localhost",
    database="postgres",  # Connect to default database
    user=usr_nm,
    password=pwd
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

# Delete the 'test' database
try:
    cur.execute(f"DROP DATABASE {NEW_DB_NAME};")
except:
    print('There are some serious troubles')
else:
    print('Database deleted sucessfully!')
finally:
    cur.close()
    conn.close()

Database deleted sucessfully!


Now you can connect to the new database!

In [37]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()

cur.execute("SELECT current_database();")
conn.commit()

#the cur.fetchone method fetches the next row of the query
print(f"Now connected to: {cur.fetchone()[0]}") 

cur.close()
conn.close()

Now connected to: test


### Creating/Deleting a table

In [54]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()

#PRIMARY KEY is to prevent from having entries with duplicate values
cur.execute("""
    CREATE TABLE IF NOT EXISTS test (
        id SERIAL PRIMARY KEY,
        num NUMERIC(10,2),
        data VARCHAR(100) NOT NULL
        );
        """
        )


#below SERIAL means that it will just increment with new data
cur.execute("""
    CREATE TABLE IF NOT EXISTS departments (
        id SERIAL,
        name VARCHAR(50) UNIQUE NOT NULL
        );
        """
        )

cur.execute("""
    CREATE TABLE IF NOT EXISTS employees (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100) NOT NULL,
        salary NUMERIC(10,2),
        dept_id INTEGER
        );
        """
        )

#commit the changes
conn.commit()

cur.close()
conn.close()

In [53]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()

try:
    cur.execute('DROP TABLE test;')
    conn.commit()
except:
    print('Error occured!')
else:
    print('Table dropped successfully!')
finally:
    cur.close()
    conn.close()

Error occured!


## Filling SQL tables

In [55]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()

#Add entries one by one into the table

#the safe way
cur.execute("INSERT INTO departments (name) VALUES (%s);", ("Engineering",))

conn.commit()

#the unsafe way
to_add = 'Sales'
cur.execute(f"INSERT INTO departments (name) VALUES ('{to_add}');")

#the 'It will result in error' way, please don't do this
try:
    cur.execute(f"INSERT INTO departments (name) VALUES ({'Sales'});") 
except Exception as e:
    print('Who would have thought? It threw an error:')
    print(str(e))

conn.commit()

#Add entries together
to_add = ((123, "foo"), (42, "bar"), (23, "baz"))
cur.executemany("INSERT INTO test (num, data) VALUES (%s, %s)", to_add)
conn.commit()


#Add entries together faster
to_add = ((546, 'Hello'), (333, 'world!'))
execute_batch(cur, "INSERT INTO test (num, data) VALUES (%s, %s)", to_add, page_size=100)

conn.commit()

cur.close()
conn.close()

Who would have thought? It threw an error:
ОШИБКА:  столбец "sales" не существует
LINE 1: INSERT INTO departments (name) VALUES (Sales);
                                               ^



## Select data from SQL

When selecting data if you just want to use it ***always*** use `LIMIT` keyword

In [56]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT * FROM test LIMIT 5;")

# Fetch all rows as a list of tuples
rows = cur.fetchall()
for row in rows:
    print(row)   # each row is a tuple, so row[0] is the name

cur.close()
conn.close()

(1, Decimal('123.00'), 'foo')
(2, Decimal('42.00'), 'bar')
(3, Decimal('23.00'), 'baz')
(4, Decimal('546.00'), 'Hello')
(5, Decimal('333.00'), 'world!')


You can also convert that data into a pandas `DataFrame`

In [71]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT * FROM test LIMIT 5;")

# Fetch all rows as a list of tuples
data = cur.fetchall()

from typing import Iterable

# print(isinstance(cur.description, Iterable))

columns = [desc[0] for desc in cur.description]
print(columns)
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

['id', 'num', 'data']
   id     num    data
0   1  123.00     foo
1   2   42.00     bar
2   3   23.00     baz
3   4  546.00   Hello
4   5  333.00  world!


## Common attributes

### ORDER BY

Same as `.sort_values` in pandas

In [72]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT * FROM test ORDER BY num ASC, data DESC;")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(columns)
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

['id', 'num', 'data']
   id     num    data
0   3   23.00     baz
1   2   42.00     bar
2   1  123.00     foo
3   5  333.00  world!
4   4  546.00   Hello


### WHERE

Apply a simple mask

In [74]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT * FROM test WHERE num < 100;")

data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(columns)
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

['id', 'num', 'data']
   id    num data
0   2  42.00  bar
1   3  23.00  baz


### GROUP BY 

Same as `pd.DataFrame.groupby`, can't be used if the columns that aren't grouped by aren't aggregated

In [86]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()

try:
    cur.execute("SELECT * FROM test GROUP BY data;")
except Exception as e:
    print(str(e))
    print("That's because you didn't use aggregation!\n\n\n")
finally:
    conn.rollback()
    # pass

cur.execute("SELECT SUM(num), data FROM test GROUP BY data;")


data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

ОШИБКА:  столбец "test.id" должен фигурировать в предложении GROUP BY или использоваться в агрегатной функции
LINE 1: SELECT * FROM test GROUP BY data;
               ^

That's because you didn't use aggregation!



      sum    data
0   42.00     bar
1  123.00     foo
2   23.00     baz
3  333.00  world!
4  546.00   Hello


### HAVING

Also used as a mask, but for `GROUP BY`

In [89]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT AVG(num), data FROM test GROUP BY data HAVING AVG(num) > 50;")


data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

                    avg    data
0  123.0000000000000000     foo
1  333.0000000000000000  world!
2  546.0000000000000000   Hello


### DISTINCT

Same as `pd.DataFrame.unique`

In [92]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT DISTINCT data FROM test;")


data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

     data
0     bar
1     foo
2     baz
3  world!
4   Hello


### JOIN

In [93]:
conn = psycopg2.connect(
    host="localhost",
    database=NEW_DB_NAME,
    user=usr_nm,
    password=pwd
)

cur = conn.cursor()


cur.execute("SELECT * FROM test INNER JOIN departments on test.id = departments.id;")


data = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

   id     num data  id         name
0   1  123.00  foo   1  Engineering


## Повторение- мать ученья

Firstly, get the series of exercises from the previous class. First, create a new db, where you'll be working
* For each exercise create the new tables (and fill them with values from the tests) and solve it via psycopg2
* After that delete all the tables

In [ ]:
###your attempts to be here